# AI-Powered Indonesian Stock Prediction & Scalping Assistant
## Modul R: AI Trading Decision Assistant (Google Colab & Standalone Jupyter)

Notebook ini mengintegrasikan **Machine Learning (LSTM & BiLSTM)**, **Analisis Teknikal Terstruktur**, **Manajemen Risiko Bursa Efek Indonesia (BEI)**, dan **Large Language Model (LLM - Gemini / OpenAI / Claude / Mock)** sebagai asisten pengambilan keputusan trading scalping.

### 1. Setup Environment & Import Dependensi

In [ ]:
# Instalasi dependensi jika dijalankan di Google Colab
try:
    import google.colab
    IN_COLAB = True
    !pip install -q yfinance pandas numpy scikit-learn torch plotly requests sqlalchemy
except ImportError:
    IN_COLAB = False

import os
import sys
import json
from datetime import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pastikan path workspace terdeteksi
workspace_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if workspace_dir not in sys.path:
    sys.path.append(workspace_dir)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from config.idx_constants import align_to_tick, get_tick_size, get_ara_arb_limits, LOT_SIZE
from src.data_collection import get_data_provider, UniverseManager
from src.technical_analysis.indicators import TechnicalAnalysisEngine
from src.machine_learning.trainer import ModelPipeline
from src.stock_screener.screener import StockScreener
from src.ai_assistant import (
    AITradingDecisionEngine,
    AIChatAssistant,
    AIDecisionValidator,
    GeminiLLMProvider,
    OpenAILLMProvider,
    ClaudeLLMProvider,
    MockLLMProvider,
    get_llm_provider
)

print("✅ Seluruh dependensi kuantitatif dan modul AI berhasil di-load!")

### 2. Konfigurasi API LLM (Mendukung Colab Secrets & Fallback Cerdas)

In [ ]:
# Mengambil API Key dari Colab Secrets atau Environment Variables
gemini_api_key = ""
openai_api_key = ""
claude_api_key = ""

if IN_COLAB:
    try:
        from google.colab import userdata
        gemini_api_key = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass
    try:
        from google.colab import userdata
        openai_api_key = userdata.get('OPENAI_API_KEY')
    except Exception:
        pass

gemini_api_key = gemini_api_key or os.getenv("GEMINI_API_KEY", "")
openai_api_key = openai_api_key or os.getenv("OPENAI_API_KEY", "")
claude_api_key = claude_api_key or os.getenv("ANTHROPIC_API_KEY", "")

# Pilih provider yang tersedia secara otomatis
if gemini_api_key:
    active_llm = GeminiLLMProvider(api_key=gemini_api_key)
    print("🤖 Menggunakan Google Gemini API")
elif openai_api_key:
    active_llm = OpenAILLMProvider(api_key=openai_api_key)
    print("🤖 Menggunakan OpenAI API")
elif claude_api_key:
    active_llm = ClaudeLLMProvider(api_key=claude_api_key)
    print("🤖 Menggunakan Anthropic Claude API")
else:
    active_llm = MockLLMProvider()
    print("⚡ Menggunakan High-Fidelity Mock LLM Provider (Offline Mode: Tidak memerlukan API key)")

# Inisialisasi Engine
provider = get_data_provider("mock") # Gunakan 'yfinance' untuk data real market .JK
ai_engine = AITradingDecisionEngine(llm_provider=active_llm)
chat_bot = AIChatAssistant(llm_provider=active_llm)

### 3. Eksekusi Analisis Kuantitatif & Keputusan AI Saham Tunggal (Single-Stock Analysis)

In [ ]:
target_ticker = "BBCA"
df_candles = provider.get_intraday_data(target_ticker, timeframe="5m", limit=80)
quote = provider.get_live_quote(target_ticker)

# Jalankan AI Decision Engine + Rule-Based Validator
ai_decision = ai_engine.analyze_stock(target_ticker, df_candles, quote, timeframe="5M")

print("=" * 60)
print(f"🎯 REKOMENDASI AI: {ai_decision.get('decision')} | VALIDASI: {ai_decision.get('validation_status')}")
print("=" * 60)
print(f"Saham              : {target_ticker}")
print(f"Harga Saat Analisis: Rp {quote.get('last_price', 0):,.0f}")
print(f"Area Entry         : {ai_decision.get('entry_zone')}")
print(f"Stop Loss (SL)     : Rp {ai_decision.get('stop_loss', 0):,.0f}")
print(f"Take Profit (TP)   : Rp {ai_decision.get('take_profit', 0):,.0f}")
print(f"Risk/Reward Ratio  : {ai_decision.get('risk_reward_ratio')}x")
print(f"Estimasi Biaya     : {ai_decision.get('estimated_transaction_cost')}")
print(f"Risiko Utama       : {ai_decision.get('main_risks')}")
print(f"Penjelasan Rinci   :\n{ai_decision.get('detailed_explanation')}")

### 4. Visualisasi Keputusan AI dengan Chart Interaktif (Plotly)

In [ ]:
df_ta = TechnicalAnalysisEngine.apply_all_indicators(df_candles)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])

# Candlestick
fig.add_trace(go.Candlestick(
    x=df_ta['timestamp'], open=df_ta['open'], high=df_ta['high'], low=df_ta['low'], close=df_ta['close'],
    name='Candlestick'
), row=1, col=1)

# Overlays: VWAP & EMA
fig.add_trace(go.Scatter(x=df_ta['timestamp'], y=df_ta['vwap'], name='VWAP', line=dict(color='#ec4899', dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_ta['timestamp'], y=df_ta['ema_20'], name='EMA 20', line=dict(color='#f59e0b')), row=1, col=1)

# Plot level TP dan SL dari AI jika BUY
if ai_decision.get('decision') == 'BUY' and ai_decision.get('validation_status') == 'APPROVED':
    fig.add_hline(y=ai_decision['take_profit'], line_dash='dash', line_color='#10b981', annotation_text='AI Take Profit', row=1, col=1)
    fig.add_hline(y=ai_decision['stop_loss'], line_dash='dash', line_color='#ef4444', annotation_text='AI Stop Loss', row=1, col=1)

# RSI 14
fig.add_trace(go.Scatter(x=df_ta['timestamp'], y=df_ta['rsi_14'], name='RSI 14', line=dict(color='#8b5cf6')), row=2, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)

fig.update_layout(title=f'Analisis Teknikal & Rekomendasi AI - {target_ticker}', height=600, template='plotly_dark')
fig.show()

### 5. AI Multi-Stock Ranking (Screening & Prioritisasi Peluang)

In [ ]:
screener = StockScreener(provider)
screened = screener.screen_stocks(min_value_idr=500_000_000, min_rel_volume=0.9)

ranking_result = ai_engine.rank_screened_stocks(screened)
print("📊 RINGKASAN PEMERINGKATAN MULTI-SAHAM OLEH AI:")
print(ranking_result.get('market_summary', ''))
print("-" * 60)
for r in ranking_result.get('ranking', []):
    print(f"[{r.get('momentum_rank')}] {r.get('ticker')}: {r.get('category')} -> {r.get('reason')}")

### 6. AI Trading Chat Assistant Interaktif

In [ ]:
# Contoh interaksi chatbot tanya-jawab
pertanyaan_user = "Mengapa saham BBCA direkomendasikan dan berapa estimasi risikonya?"
jawaban_ai = chat_bot.answer_query(
    pertanyaan_user,
    current_market_context=ai_decision,
    screened_stocks=screened
)

print(f"👤 User: {pertanyaan_user}\n")
print(f"🤖 AI Assistant:\n{jawaban_ai}")